# PlateGuard MBG — Training Lengkap di Kaggle (Sesuai Project Charter)

Notebook ini mengadaptasi [`train.py`](https://github.com/cupiz/datascience-mbg-eda/blob/main/train.py) agar seluruh pipeline pelatihan dapat dijalankan interaktif di **Kaggle Notebooks** atau **Google Colab** lengkap dengan output inline.

### Yang Dicakup Notebook Ini (Checklist Charter)
1. **Guardrail Anti-Kebocoran Data:** Eliminasi `waste_adjusted_effective_cost_idr`, `bgn_program_compliance_tier`, `confidence_score`, narasi CoT, dan `distribution_id`.
2. **Feature Engineering + `province` Target-Encoding:** `ReflectionLog`, `Log1p`, `QuantileBinner`, fitur rasio & interaksi, ditambah `TargetEncoder` untuk 38 provinsi (`34` fitur pra / `38` fitur post-hoc) yang di-*fit* ketat hanya pada lipatan latih.
3. **6 Keluarga Algoritma Charter:** `MeanPredictor`, `Ridge`, **`RandomForest`**, `HistGradientBoosting`, `LightGBM`, dan `XGBoost`.
4. **5-Fold CV Per-Fold + Optuna (30 Trial):** Laporan lengkap Fold 1 s.d. Fold 5 (MAE, RMSE, $R^2$) dan evaluasi akhir pada 2.600 data uji holdout.
5. **Explainable AI (TreeSHAP) + MLflow + Paket Unduh Kaggle:** Grafik SHAP inline, pelacakan eksperimen `mlruns.db`, model `.joblib` & `.pkl`, serta arsip `plateguard_kaggle_outputs.zip` di `/kaggle/working/`.

### Cara Menjalankan di Kaggle
1. Buat Notebook baru di Kaggle, unggah berkas `train_kaggle.ipynb` ini.
2. Pada panel kanan: aktifkan **Settings → Internet: On** (akselerator CPU sudah cukup cepat, ~2–5 menit).
3. Klik **Run All**. Setelah selesai, unduh **`plateguard_kaggle_outputs.zip`** dari tab **Output** lalu ekstrak ke folder `Proyek_MBG/` lokal.

In [1]:
# Sel 1 — Setup Repositori & Dependensi (Kaggle / Colab / Lokal)
%matplotlib inline
import os
import subprocess
import sys

os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

REPO_URL = "https://github.com/cupiz/datascience-mbg-eda.git"
REPO_DIR = "datascience-mbg-eda"

# Deteksi apakah notebook dijalankan langsung dari dalam folder proyek atau di cloud (/kaggle/working)
if os.path.exists("train.py") and os.path.exists("etl_pipeline.py"):
    print("Berjalan langsung di direktori repositori lokal:", os.getcwd())
else:
    if os.path.isdir(REPO_DIR):
        os.chdir(REPO_DIR)
        subprocess.run(["git", "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
        os.chdir(REPO_DIR)

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# Pastikan scikit-learn >= 1.4 untuk TargetEncoder + pustaka ML/MLflow tersedia
import sklearn
sk_ver = tuple(int(p) for p in sklearn.__version__.split(".")[:2])
reqs = []
if sk_ver < (1, 4):
    reqs.append("scikit-learn>=1.4.0")

for pkg, pip_name in [
    ("xgboost", "xgboost"),
    ("lightgbm", "lightgbm"),
    ("optuna", "optuna"),
    ("shap", "shap"),
    ("mlflow", "mlflow"),
    ("kagglehub", "kagglehub"),
    ("joblib", "joblib"),
]:
    try:
        __import__(pkg)
    except ImportError:
        reqs.append(pip_name)

if reqs:
    print("Memasang paket tambahan:", reqs)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *reqs], check=True)

print("Siap! Working directory:", os.getcwd())

Cloning into 'datascience-mbg-eda'...


Memasang paket tambahan: ['mlflow']
Siap! Working directory: /kaggle/working/datascience-mbg-eda


### Langkah 1 — Akuisisi Dataset & Guardrail Kebocoran Data

Karena file CSV mentah tidak di-commit ke GitHub (`data/*.csv` masuk `.gitignore`), sel berikut otomatis mengecek input Kaggle maupun mengunduh dataset publik `luthfimuthathohirin/indonesia-makan-bergizi-gratis-mbg-telemetry` via `kagglehub`, lalu menerapkan pembersihan kolom bocor lewat fungsi `load_clean_raw()` di `train.py`.

In [2]:
# Sel 2 — Unduh Dataset Otomatis & Validasi Guardrail Awal
import glob
import shutil
import pandas as pd
import etl_pipeline
import train

RAW_PATH = etl_pipeline.RAW
os.makedirs(os.path.dirname(RAW_PATH), exist_ok=True)

if not os.path.exists(RAW_PATH):
    # Cek apakah dataset sudah ditambahkan lewat Kaggle Add Input (/kaggle/input)
    kaggle_matches = glob.glob("/kaggle/input/**/indonesia_mbg_nationwide_enriched_2026.csv", recursive=True)
    if kaggle_matches:
        shutil.copy(kaggle_matches[0], RAW_PATH)
        print("Dataset disalin dari Kaggle Input:", kaggle_matches[0])
    else:
        import kagglehub
        dl_dir = kagglehub.dataset_download("luthfimuthathohirin/indonesia-makan-bergizi-gratis-mbg-telemetry")
        src_csv = os.path.join(dl_dir, "indonesia_mbg_nationwide_enriched_2026.csv")
        if not os.path.exists(src_csv):
            src_csv = glob.glob(os.path.join(dl_dir, "*.csv"))[0]
        shutil.copy(src_csv, RAW_PATH)
        print("Dataset diunduh via kagglehub:", src_csv)
else:
    print("Dataset lokal sudah tersedia:", RAW_PATH)

df_clean = train.load_clean_raw()
print(f"Dimensi pasca-guardrail: {df_clean.shape[0]:,} baris x {df_clean.shape[1]} kolom")
print("Kolom bocor yang dibuang:", etl_pipeline.DROP_LEAK)
print(f"Provinsi unik: {df_clean['province'].nunique()} | Null: {int(df_clean.isna().sum().sum())}")
display(df_clean.head(3))

Dataset diunduh via kagglehub: /kaggle/input/datasets/luthfimuthathohirin/indonesia-makan-bergizi-gratis-mbg-telemetry/indonesia_mbg_nationwide_enriched_2026.csv
Dimensi pasca-guardrail: 13,000 baris x 21 kolom
Kolom bocor yang dibuang: ['waste_adjusted_effective_cost_idr', 'bgn_program_compliance_tier', 'confidence_score', 'cot_nutrition_logistics_rationale', 'distribution_id']
Provinsi unik: 38 | Null: 0


,province,regional_cluster,beneficiary_category,kitchen_daily_capacity_portions,raw_ingredient_cost_per_portion_idr,protein_type,delivery_distance_km,delivery_transport_mode,transit_time_minutes,food_arrival_temperature_celsius,...,student_attendance_rate_pct,local_farmer_sourcing_pct,total_meal_cost_idr,estimated_calories_kcal,estimated_protein_grams,transit_cooling_rate_c_per_min,spoilage_risk_index,cost_to_nutrition_ratio,local_economic_multiplier_score,primary_food_waste_driver
0,Aceh,Daerah Pedesaan & Kepulauan,Balita & Ibu Hamil Posyandu,1422,14500,Daging Sapi / Kerbau,24.5,Mobil Pickup Termal,77,56.9,...,91.9,73.9,15000.0,665.0,29.1,0.235,4.57,515.5,7.26,Delivery-Transit-Delay
1,Aceh,Daerah Pedesaan & Kepulauan,Siswa SD / Madrasah Ibtidaiyah,1449,11900,Daging Ayam Broiler,3.0,Perahu Ketinting Sungai / Laut,18,65.5,...,90.5,71.2,14335.0,608.0,23.0,0.528,3.02,623.3,7.04,Prior-Snack-Consumption
2,Aceh,Daerah Pedesaan & Kepulauan,Siswa SD / Madrasah Ibtidaiyah,1133,12200,Daging Ayam Broiler,3.1,Mobil Pickup Termal,12,68.0,...,97.1,77.0,14440.0,614.0,23.2,0.583,0.75,622.4,7.51,Prior-Snack-Consumption


### Langkah 2 — Konfigurasi Eksperimen, Stratified Split 80:20 & Transformasi Fitur + `province` Target-Encoding

- `train_df` (`10.400` baris) dan `test_df` (`2.600` baris) dipartisi secara deterministik berdasarkan kuartil target (`random_state=42`).
- Seluruh transformator (`ReflectionLog`, `Log1p`, `QuantileBinner`, `OneHotEncoder`, **`TargetEncoder(province)`**, dan `StandardScaler`) di-*fit* secara terisolasi pada tiap lipatan latih (`5-fold CV`) agar tidak ada kebocoran target ke data validasi maupun uji.

In [3]:
# Sel 3 — Konfigurasi Pelatihan & Persiapan 5-Fold CV + Matriks Holdout
import time

N_TRIALS = 30          # Ubah ke 15 jika ingin run lebih singkat
CV_FOLDS = 5           # Sesuai target validasi silang 5-fold Charter
USE_PROVINCE_TE = True # Target-encoding 38 provinsi

train_df, test_df = etl_pipeline.stratified_split(df_clean, test_size=0.2, random_state=train.RANDOM_STATE)
X_train_raw = train_df.drop(columns=[etl_pipeline.TARGET])
y_train = train_df[etl_pipeline.TARGET]
X_test_raw = test_df.drop(columns=[etl_pipeline.TARGET])
y_test = test_df[etl_pipeline.TARGET]

print(f"Train: {X_train_raw.shape} (target mean={y_train.mean():.3f}%) | Test Holdout: {X_test_raw.shape} (target mean={y_test.mean():.3f}%)")

folds_by_scenario = {}
holdout_data = {}

for sc in ("pra", "posthoc"):
    t0 = time.time()
    folds = train.prepare_cv_folds(X_train_raw, y_train, sc, USE_PROVINCE_TE, CV_FOLDS)
    full_pipe = etl_pipeline.build_pipeline(sc, use_province_te=USE_PROVINCE_TE)
    X_tr_arr = full_pipe.fit_transform(X_train_raw, y_train.to_numpy(dtype=float))
    feat_cols = list(full_pipe[-2].get_feature_names_out())
    X_tr_df = pd.DataFrame(X_tr_arr, columns=feat_cols)
    X_te_df = pd.DataFrame(full_pipe.transform(X_test_raw), columns=feat_cols)
    train.verify_transformed_guardrail(X_tr_df, X_te_df)

    folds_by_scenario[sc] = folds
    holdout_data[sc] = (full_pipe, X_tr_df, X_te_df, feat_cols)
    print(f"[{sc.upper():7s}] siap dalam {time.time()-t0:.2f}s | jumlah fitur = {len(feat_cols)} | TE province aktif: {'prov_te__province' in feat_cols}")

Train: (10400, 20) (target mean=13.331%) | Test Holdout: (2600, 20) (target mean=13.350%)
[PRA    ] siap dalam 0.32s | jumlah fitur = 34 | TE province aktif: True
[POSTHOC] siap dalam 0.34s | jumlah fitur = 38 | TE province aktif: True


### Langkah 3 — Uji Ablasi Singkat: Pengaruh `province` Target-Encoding

Sebelum menjalankan benchmark seluruh model, sel ini membandingkan kinerja 5-fold CV `XGBoost` default **tanpa** target-encoding provinsi (`33/37` fitur) vs **dengan** target-encoding provinsi (`34/38` fitur).

In [4]:
# Sel 4 — Ablasi Tanpa vs Dengan Province Target-Encoding
from sklearn.base import clone
from xgboost import XGBRegressor

ablation_rows = []
xgb_probe = XGBRegressor(n_estimators=400, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, random_state=42, n_jobs=-1)

for sc in ("pra", "posthoc"):
    folds_no_te = train.prepare_cv_folds(X_train_raw, y_train, sc, use_province_te=False, cv_folds=CV_FOLDS)
    _, sum_no = train.evaluate_model_cv("XGBoost", xgb_probe, folds_no_te)
    _, sum_yes = train.evaluate_model_cv("XGBoost", xgb_probe, folds_by_scenario[sc])
    ablation_rows.append({
        "Skenario": sc, "Konfigurasi": f"Tanpa Province TE ({folds_no_te[0][0].shape[1]} fitur)",
        "CV MAE": sum_no["cv_mae_mean"], "CV RMSE": sum_no["cv_rmse_mean"], "CV R2": sum_no["cv_r2_mean"],
    })
    ablation_rows.append({
        "Skenario": sc, "Konfigurasi": f"Dengan Province TE ({folds_by_scenario[sc][0][0].shape[1]} fitur)",
        "CV MAE": sum_yes["cv_mae_mean"], "CV RMSE": sum_yes["cv_rmse_mean"], "CV R2": sum_yes["cv_r2_mean"],
    })

display(pd.DataFrame(ablation_rows))

,Skenario,Konfigurasi,CV MAE,CV RMSE,CV R2
0,pra,Tanpa Province TE (33 fitur),3.7115,4.6377,0.3074
1,pra,Dengan Province TE (34 fitur),3.7102,4.6376,0.3075
2,posthoc,Tanpa Province TE (37 fitur),2.9893,3.8217,0.5297
3,posthoc,Dengan Province TE (38 fitur),2.9840,3.8139,0.5316


### Langkah 4 — Benchmark 6 Algoritma Charter & Laporan 5-Fold CV Per-Fold

Mengevaluasi `MeanPredictor`, `Ridge`, **`RandomForest`**, `HistGradientBoosting`, `LightGBM`, dan `XGBoost` pada setiap lipatan Fold 1 s.d. Fold 5 serta pada holdout test set (2.600 baris).

In [5]:
# Sel 5 — Benchmark 6 Kandidat Model & Rekap Metrik Per-Fold
import numpy as np

candidates = train.make_base_candidates()
all_cv_rows = []
holdout_records = {sc: [] for sc in ("pra", "posthoc")}
cv_summaries = {sc: {} for sc in ("pra", "posthoc")}
base_records_for_mlflow = {sc: [] for sc in ("pra", "posthoc")}

for sc in ("pra", "posthoc"):
    folds = folds_by_scenario[sc]
    _, X_tr_full, X_te_full, feat_cols = holdout_data[sc]
    print(f"\n=== SKENARIO {sc.upper()} ({len(feat_cols)} fitur) ===")

    for name, est in candidates.items():
        fold_rows, cv_sum = train.evaluate_model_cv(name, est, folds)
        for row in fold_rows:
            all_cv_rows.append({"scenario": sc, **row})
        cv_summaries[sc][name] = cv_sum

        model_full = clone(est)
        model_full.fit(X_tr_full, y_train)
        h_metrics = train.calc_metrics(y_test.to_numpy(dtype=float), model_full.predict(X_te_full))

        holdout_records[sc].append({
            "model": name, **cv_sum,
            "holdout_MAE": h_metrics["MAE"], "holdout_RMSE": h_metrics["RMSE"], "holdout_R2": h_metrics["R2"],
        })
        base_records_for_mlflow[sc].append((name, model_full, fold_rows, cv_sum, h_metrics))
        print(f"  {name:22s} CV MAE={cv_sum['cv_mae_mean']:.3f}±{cv_sum['cv_mae_std']:.3f} -> Holdout MAE={h_metrics['MAE']:.3f} RMSE={h_metrics['RMSE']:.3f} R2={h_metrics['R2']:.3f}")

df_cv_base = pd.DataFrame(all_cv_rows)
print("\nContoh Ringkasan Per-Fold (Fold 1..5) Model Base:")
display(df_cv_base.pivot_table(index=["scenario", "model"], columns="fold", values="MAE").round(3))


=== SKENARIO PRA (34 fitur) ===
  MeanPredictor          CV MAE=4.443±0.043 -> Holdout MAE=4.447 RMSE=5.613 R2=-0.000
  Ridge                  CV MAE=3.683±0.037 -> Holdout MAE=3.720 RMSE=4.672 R2=0.307
  RandomForest           CV MAE=3.636±0.040 -> Holdout MAE=3.658 RMSE=4.604 R2=0.327
  HistGradientBoosting   CV MAE=3.792±0.036 -> Holdout MAE=3.660 RMSE=4.592 R2=0.331
  LightGBM               CV MAE=3.687±0.034 -> Holdout MAE=3.706 RMSE=4.647 R2=0.315
  XGBoost                CV MAE=3.710±0.034 -> Holdout MAE=3.712 RMSE=4.665 R2=0.309

=== SKENARIO POSTHOC (38 fitur) ===
  MeanPredictor          CV MAE=4.443±0.043 -> Holdout MAE=4.447 RMSE=5.613 R2=-0.000
  Ridge                  CV MAE=2.997±0.033 -> Holdout MAE=3.018 RMSE=3.892 R2=0.519
  RandomForest           CV MAE=2.943±0.038 -> Holdout MAE=2.962 RMSE=3.797 R2=0.542
  HistGradientBoosting   CV MAE=3.046±0.026 -> Holdout MAE=2.940 RMSE=3.773 R2=0.548
  LightGBM               CV MAE=2.964±0.032 -> Holdout MAE=2.977 RMSE=3.819 R2

fold                               1      2      3      4      5
scenario model                                                  
posthoc  HistGradientBoosting  3.047  3.007  3.027  3.070  3.077
         LightGBM              2.952  2.922  2.944  2.994  3.009
         MeanPredictor         4.447  4.448  4.380  4.514  4.424
         RandomForest          2.932  2.895  2.914  2.980  2.993
         Ridge                 3.024  2.968  2.950  3.006  3.036
         XGBoost               2.966  2.930  2.974  3.031  3.019
pra      HistGradientBoosting  3.730  3.826  3.772  3.820  3.811
         LightGBM              3.634  3.708  3.661  3.718  3.715
         MeanPredictor         4.447  4.448  4.380  4.514  4.424
         RandomForest          3.577  3.658  3.601  3.668  3.677
         Ridge                 3.639  3.719  3.645  3.683  3.730
         XGBoost               3.659  3.756  3.686  3.733  3.716

### Langkah 5 — Optimasi Bayesian Optuna (`30 Trial x 5-Fold CV`)

Memilih booster terbaik per skenario (`XGBoost` vs `LightGBM`), melakukan optimasi Bayesian pada ruang parameter yang dipagari di lipatan 5-Fold CV, lalu mengevaluasi model final satu kali pada 2.600 data holdout dan memeriksa ketercapaian target Project Charter (`MAE <= 3,0`, `RMSE <= 4,5`, `R2 >= 0,35`).

In [6]:
# Sel 6 — Optuna Bayesian Hyperparameter Tuning & Evaluasi Final Holdout
best_bundles = {}
final_results = {}

for sc in ("pra", "posthoc"):
    folds = folds_by_scenario[sc]
    full_pipe, X_tr_full, X_te_full, feat_cols = holdout_data[sc]
    booster_choice = min(["XGBoost", "LightGBM"], key=lambda m: cv_summaries[sc][m]["cv_mae_mean"])
    print(f"\n[{sc.upper()}] Menjalankan Optuna ({N_TRIALS} trial x {CV_FOLDS}-Fold) untuk {booster_choice}...")

    t_opt = time.time()
    tuned_est, best_params = train.tune_top_booster_optuna(booster_choice, folds, n_trials=N_TRIALS)
    tuned_name = f"{booster_choice} (Optuna)"

    opt_fold_rows, opt_cv_sum = train.evaluate_model_cv(tuned_name, tuned_est, folds)
    for row in opt_fold_rows:
        all_cv_rows.append({"scenario": sc, **row})

    best_model = clone(tuned_est)
    best_model.fit(X_tr_full, y_train)
    opt_holdout = train.calc_metrics(y_test.to_numpy(dtype=float), best_model.predict(X_te_full))

    holdout_records[sc].append({
        "model": tuned_name, **opt_cv_sum,
        "holdout_MAE": opt_holdout["MAE"], "holdout_RMSE": opt_holdout["RMSE"], "holdout_R2": opt_holdout["R2"],
    })
    holdout_sorted = sorted(holdout_records[sc], key=lambda r: r["holdout_MAE"])
    best_bundles[sc] = (tuned_name, best_model, best_params, opt_fold_rows, opt_cv_sum, opt_holdout)

    final_results[sc] = {
        "scenario": sc,
        "best_model": f"[{sc}] {tuned_name}",
        "MAE": round(opt_holdout["MAE"], 3),
        "RMSE": round(opt_holdout["RMSE"], 3),
        "R2": round(opt_holdout["R2"], 3),
        "target_MAE<=3.0": bool(opt_holdout["MAE"] <= 3.0),
        "target_R2>=0.35": bool(opt_holdout["R2"] >= 0.35),
        "target_RMSE<=4.5": bool(opt_holdout["RMSE"] <= 4.5),
        "n_features": len(feat_cols),
        "use_province_te": USE_PROVINCE_TE,
        "best_params": best_params,
        "cv_5fold_summary": opt_cv_sum,
        "cv_per_fold": opt_fold_rows,
        "holdout_all_models": holdout_sorted,
    }
    print(f"Selesai ({time.time()-t_opt:.1f}s) | Best Params: {best_params}")
    display(pd.DataFrame(holdout_sorted))
    print("Status Target Charter:", {k: final_results[sc][k] for k in ["best_model", "MAE", "RMSE", "R2", "target_MAE<=3.0", "target_R2>=0.35", "target_RMSE<=4.5"]})


[PRA] Menjalankan Optuna (30 trial x 5-Fold) untuk LightGBM...
Selesai (120.4s) | Best Params: {'n_estimators': 320, 'learning_rate': 0.013272464743316144, 'subsample': 0.6781409707063616, 'colsample_bytree': 0.661679266915142, 'reg_alpha': 0.47694154050889437, 'reg_lambda': 0.0015628484037241322, 'num_leaves': 15, 'min_child_samples': 27}


,model,cv_mae_mean,cv_mae_std,cv_rmse_mean,cv_rmse_std,cv_r2_mean,cv_r2_std,holdout_MAE,holdout_RMSE,holdout_R2
0,RandomForest,3.6361,0.0396,4.5505,0.0290,0.3332,0.0084,3.6575,4.6036,0.3272
1,LightGBM (Optuna),3.6044,0.0417,4.5096,0.0302,0.3452,0.0076,3.6583,4.5856,0.3325
2,HistGradientBoosting,3.7920,0.0362,4.7367,0.0230,0.2776,0.0037,3.6604,4.5923,0.3305
3,LightGBM,3.6874,0.0336,4.6113,0.0273,0.3153,0.0057,3.7058,4.6467,0.3146
4,XGBoost,3.7102,0.0342,4.6376,0.0306,0.3075,0.0071,3.7117,4.6650,0.3092
5,Ridge,3.6832,0.0372,4.6193,0.0341,0.3129,0.0116,3.7205,4.6720,0.3071
6,MeanPredictor,4.4427,0.0435,5.5764,0.0309,-0.0013,0.0015,4.4469,5.6127,-0.0000


Status Target Charter: {'best_model': '[pra] LightGBM (Optuna)', 'MAE': 3.658, 'RMSE': 4.586, 'R2': 0.333, 'target_MAE<=3.0': False, 'target_R2>=0.35': False, 'target_RMSE<=4.5': False}

[POSTHOC] Menjalankan Optuna (30 trial x 5-Fold) untuk LightGBM...
Selesai (140.9s) | Best Params: {'n_estimators': 350, 'learning_rate': 0.010428253596958125, 'subsample': 0.769810490763503, 'colsample_bytree': 0.6399125626413396, 'reg_alpha': 0.9235784336659716, 'reg_lambda': 0.006246222037806818, 'num_leaves': 29, 'min_child_samples': 14}


,model,cv_mae_mean,cv_mae_std,cv_rmse_mean,cv_rmse_std,cv_r2_mean,cv_r2_std,holdout_MAE,holdout_RMSE,holdout_R2
0,LightGBM (Optuna),2.9051,0.0464,3.7099,0.0348,0.5568,0.0064,2.9295,3.7613,0.5509
1,HistGradientBoosting,3.0457,0.0261,3.8902,0.0262,0.5127,0.0049,2.9401,3.7728,0.5482
2,RandomForest,2.9429,0.0377,3.7540,0.0305,0.5462,0.0057,2.9616,3.7969,0.5424
3,LightGBM,2.9642,0.0323,3.7783,0.0328,0.5403,0.0062,2.9774,3.8192,0.5370
4,XGBoost,2.9840,0.0366,3.8139,0.0409,0.5316,0.0070,2.9790,3.8061,0.5401
5,Ridge,2.9969,0.0327,3.8471,0.0154,0.5234,0.0044,3.0181,3.8920,0.5191
6,MeanPredictor,4.4427,0.0435,5.5764,0.0309,-0.0013,0.0015,4.4469,5.6127,-0.0000


Status Target Charter: {'best_model': '[posthoc] LightGBM (Optuna)', 'MAE': 2.929, 'RMSE': 3.761, 'R2': 0.551, 'target_MAE<=3.0': True, 'target_R2>=0.35': True, 'target_RMSE<=4.5': True}


### Langkah 6 — Interpretabilitas Explainable AI (TreeSHAP Inline)

Menghitung nilai Shapley pada 2.600 sampel uji holdout untuk kedua model terbaik, menyimpan ke `figures_fase3/`, dan menampilkan grafik Beeswarm + Feature Importance langsung di sel output.

In [7]:
# Sel 7 — Visualisasi & Penyimpanan Plot TreeSHAP
import matplotlib.pyplot as plt
import shap

os.makedirs("figures_fase3", exist_ok=True)
shap_files_by_sc = {}

for sc in ("pra", "posthoc"):
    tuned_name, best_model, _, _, _, _ = best_bundles[sc]
    _, _, X_te_full, _ = holdout_data[sc]
    explainer = shap.TreeExplainer(best_model)
    sv = explainer.shap_values(X_te_full)

    bee_path = f"figures_fase3/shap_beeswarm_{sc}.png"
    plt.figure(figsize=(10, 6))
    shap.summary_plot(sv, X_te_full, show=False, max_display=15)
    plt.title(f"SHAP Beeswarm - Skenario {sc.upper()} ({tuned_name})")
    plt.tight_layout()
    plt.savefig(bee_path, dpi=150)
    plt.show()
    plt.close()

    bar_path = f"figures_fase3/shap_bar_{sc}.png"
    plt.figure(figsize=(10, 6))
    shap.summary_plot(sv, X_te_full, plot_type="bar", show=False, max_display=15)
    plt.title(f"SHAP Feature Importance - Skenario {sc.upper()} ({tuned_name})")
    plt.tight_layout()
    plt.savefig(bar_path, dpi=150)
    plt.show()
    plt.close()

    shap_files_by_sc[sc] = [bee_path, bar_path]
    print(f"[{sc.upper()}] SHAP disimpan:", bee_path, "dan", bar_path)

[PRA] SHAP disimpan: figures_fase3/shap_beeswarm_pra.png dan figures_fase3/shap_bar_pra.png
[POSTHOC] SHAP disimpan: figures_fase3/shap_beeswarm_posthoc.png dan figures_fase3/shap_bar_posthoc.png


### Langkah 7 — Logging MLflow, Ekspor Artefak (`.joblib` + `.pkl` + CSV + JSON), & Arsip Zip Kaggle

Sel terakhir mencatat seluruh riwayat eksperimen ke MLflow (`mlruns.db`), menyimpan artefak yang langsung kompatibel dengan `api.py` dan `app.py`, lalu membungkus semuanya menjadi `plateguard_kaggle_outputs.zip` di `/kaggle/working/` agar mudah diunduh.

In [15]:
import importlib, os, shutil, subprocess, mlflow
if mlflow.active_run() is not None:
    mlflow.end_run()
if os.path.exists("mlruns.db"):
    os.remove("mlruns.db")
shutil.rmtree("mlruns", ignore_errors=True)
subprocess.run(["git", "pull", "--ff-only"], check=True)
importlib.reload(train)
print("Fix berhasil dimuat! Silakan jalankan Sel 8.")

Updating cfb2a19..4214a0e
Fast-forward
 api.py                        | 12 ++++++++----
 app.py                        | 11 ++++++++---
 make_train_kaggle_notebook.py |  4 ++++
 train.py                      | 15 +++++++++++++--
 train_kaggle.ipynb            |  4 ++++
 5 files changed, 37 insertions(+), 9 deletions(-)
Fix berhasil dimuat! Silakan jalankan Sel 8.


From https://github.com/cupiz/datascience-mbg-eda
   cfb2a19..4214a0e  main       -> origin/main


In [18]:
import json, os, pickle, zipfile, joblib, mlflow

if mlflow.active_run() is not None:
    mlflow.end_run()

DB_FILE = os.path.abspath("mlruns_final.db")
mlflow.set_tracking_uri(f"sqlite:///{DB_FILE}")
mlflow.set_experiment("plateguard-mbg-training")

os.makedirs("models", exist_ok=True)
for sc in ("pra", "posthoc"):
    full_pipe, X_tr_full, X_te_full, feat_cols = holdout_data[sc]
    tuned_name, best_model, best_params, opt_fold_rows, opt_cv_sum, opt_holdout = best_bundles[sc]

    proc_dir = f"data/processed/{sc}"
    os.makedirs(proc_dir, exist_ok=True)
    X_tr_full.round(4).to_csv(f"{proc_dir}/X_train.csv", index=False)
    X_te_full.round(4).to_csv(f"{proc_dir}/X_test.csv", index=False)
    y_train.to_csv(f"{proc_dir}/y_train.csv", index=False)
    y_test.to_csv(f"{proc_dir}/y_test.csv", index=False)

    joblib.dump(full_pipe, f"models/pipeline_{sc}.joblib")
    bundle = {
        "name": f"[{sc}] {tuned_name}",
        "model": best_model,
        "params": best_params,
        "features": feat_cols,
        "use_province_te": USE_PROVINCE_TE,
        "cv_5fold_summary": opt_cv_sum,
        "cv_per_fold": opt_fold_rows,
        "holdout_metrics": opt_holdout,
    }
    joblib.dump(bundle, f"models/best_{sc}.joblib")
    with open(f"models/best_{sc}.pkl", "wb") as fp:
        pickle.dump(bundle, fp)

    for m_name, m_obj, f_rows, cv_sum, h_met in base_records_for_mlflow[sc]:
        raw_p = m_obj.get_params() if hasattr(m_obj, "get_params") else {}
        simple_p = {k: v for k, v in raw_p.items() if isinstance(v, (int, float, str, bool)) or v is None}
        train.log_mlflow_run(sc, m_name, simple_p, f_rows, cv_sum, h_met, USE_PROVINCE_TE, len(feat_cols))

    train.log_mlflow_run(
        sc, tuned_name, best_params, opt_fold_rows, opt_cv_sum, opt_holdout,
        USE_PROVINCE_TE, len(feat_cols), fitted_model=best_model, extra_artifacts=shap_files_by_sc[sc]
    )

pd.DataFrame(all_cv_rows).to_csv("models/cv_per_fold_report.csv", index=False)
with open("models/fase3_metrics.json", "w", encoding="utf-8") as fh:
    json.dump(final_results, fh, indent=1, ensure_ascii=False)

out_root = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
zip_path = os.path.join(out_root, "plateguard_kaggle_outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for d in ["models", "figures_fase3", "data/processed", "mlruns"]:
        if os.path.exists(d):
            for root_dir, _, files in os.walk(d):
                for fn in files:
                    fp = os.path.join(root_dir, fn)
                    zf.write(fp, arcname=os.path.relpath(fp, "."))
    if os.path.exists(DB_FILE):
        zf.write(DB_FILE, arcname="mlruns.db")

print("SUKSES! Ringkasan MLflow runs:", mlflow.search_runs(experiment_names=["plateguard-mbg-training"]).shape[0], "runs tercatat.")
print("Zip hasil siap diunduh di:", zip_path, f"({os.path.getsize(zip_path)/1e6:.2f} MB)")

2026/11/17 18:16:16 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/17 18:16:16 INFO mlflow.store.db.utils: Updating database tables
2026/11/17 18:16:19 INFO mlflow.tracking.fluent: Experiment with name 'plateguard-mbg-training' does not exist. Creating a new experiment.
2026/11/17 18:16:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/11/17 18:16:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative 

SUKSES! Ringkasan MLflow runs: 14 runs tercatat.
Zip hasil siap diunduh di: /kaggle/working/plateguard_kaggle_outputs.zip (4.20 MB)


In [14]:
import torch

print("GPU tersedia:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nama GPU:", torch.cuda.get_device_name(0))
    print("Jumlah GPU:", torch.cuda.device_count())


GPU tersedia: False
